# Episodata: from schema to training batch

A runnable companion to the [getting-started guide](../docs/getting_started.md).
This notebook starts before any data exists, builds a small dataset online,
and follows it through reads and batched sampling. The main path uses only
Episodata and NumPy; framework integration is optional.

In [ ]:
import numpy as np

from episodata import Dataset, DatasetSchema, FieldSpec

## 1. Define the data model

A schema declares logical fields independently of storage. Shapes are per
step: they contain neither a time nor a batch dimension. Nested structures
use stable `/`-separated keys.

In [ ]:
schema = DatasetSchema([
    FieldSpec("agent/position", shape=(2,), dtype="float32"),
    FieldSpec("agent/velocity", shape=(2,), dtype="float32"),
    FieldSpec("action", shape=(2,), dtype="float32", role="action"),
    FieldSpec("reward", shape=(), dtype="float32", role="reward"),
])

dataset = Dataset.create(schema)  # memory backend when no path is given
dataset

## 2. Collect episodes online

`new_episode()` records the reset observation. Each `add_step()` then records
one environment transition. A true `terminated` or `truncated` value
finalizes the episode.

In [ ]:
def collect_episode(dataset, length, start=0.0):
    position = np.array([start, 0.0], dtype=np.float32)
    velocity = np.array([1.0, 0.25], dtype=np.float32)
    writer = dataset.new_episode({
        "agent": {"position": position, "velocity": velocity}
    })

    for step in range(length):
        position = position + velocity
        writer.add_step({
            "observations": {
                "agent": {"position": position, "velocity": velocity}
            },
            "actions": velocity,
            "rewards": np.float32(1.0),
            "terminated": step == length - 1,
        })

    return dataset.episode(writer.episode_id)


long_episode = collect_episode(dataset, length=6)
short_episode = collect_episode(dataset, length=3, start=10.0)
list(dataset.episodes())

## 3. Read transitions, not storage rows

An episode with `T` steps has length `T`. `segment(start, stop)` returns the
transitions in `[start, stop)`: the observation where each action was taken,
the action and reward, and the observation it produced.

In [ ]:
segment = long_episode.segment(1, 4)

{
    "episode_length": len(long_episode),
    "observation": segment.obs.agent.position,
    "action": segment.action,
    "next_observation": segment.next_obs.agent.position,
    "terminated": segment.terminated,
}

`obs` and `next_obs` are overlapping views of `all_obs`, the `L + 1`
observations spanned by the segment. Nested attribute access reconstructs
the hierarchy; bracket access always accepts the stable flat key.

In [ ]:
positions = segment.all_obs["agent/position"]

assert np.array_equal(segment.obs.agent.position, positions[:-1])
assert np.array_equal(segment.next_obs.agent.position, positions[1:])
assert np.shares_memory(segment.obs.agent.position, segment.next_obs.agent.position)

positions

## 4. Build an indexable training view

`segments()` creates a stable map-style view over all valid windows.
`fetch(indices)` resolves many windows and returns one `Batch` through a
single batched backend read. The short episode contributes one padded
window, with `mask=False` on padding.

In [ ]:
segments = dataset.segments(sequence_length=4)
batch = segments.fetch([0, len(segments) - 1])

{
    "number_of_windows": len(segments),
    "observation_shape": batch.obs.agent.position.shape,
    "action_shape": batch.action.shape,
    "mask": batch.mask,
}

## 5. Sample context and target sequences

`segment_stream()` samples windows uniformly with replacement by default.
Context and target are contiguous transition views of the same batch. The
stream reads larger chunks internally and buffers ready batches.

In [ ]:
stream = dataset.segment_stream(
    context_length=2,
    target_length=2,
    batch_size=3,
    seed=0,
    read_chunk_size=6,
)
batch = stream.sample()

assert np.array_equal(
    np.concatenate([batch.context.obs.agent.position, batch.target.obs.agent.position], axis=1),
    batch.obs.agent.position,
)

{
    "batch": batch.obs.agent.position.shape,
    "context": batch.context.obs.agent.position.shape,
    "target": batch.target.obs.agent.position.shape,
}

Sampling policy is separate from data access. A custom sampler returns flat
indices into the supplied segment index; `SegmentDataset` handles the reads,
padding, and batch construction.

In [ ]:
class AlwaysLastSampler:
    def sample(self, index, batch_size):
        return np.full(batch_size, len(index) - 1, dtype=np.int64)


short_batches = dataset.segment_stream(
    sequence_length=4,
    batch_size=2,
    sampler=AlwaysLastSampler(),
    read_chunk_size=2,
)
short_batches.sample().mask

## 6. Optional PyTorch integration

`SegmentDataset` follows the map-style DataLoader protocol and exposes a
batched-fetch hook. Episodata does not require or import PyTorch; this cell
runs only when PyTorch is installed. `map()` converts each underlying buffer
once, preserving the shared storage between `obs` and `next_obs`.

In [ ]:
try:
    import torch
    from torch.utils.data import DataLoader
except ImportError:
    print("Optional example skipped: install torch to run it.")
else:
    loader = DataLoader(
        segments,
        batch_size=2,
        shuffle=True,
        collate_fn=segments.collate,
    )
    torch_batch = next(iter(loader)).map(torch.as_tensor)
    print(torch_batch.obs.agent.position.shape, torch_batch.obs.agent.position.dtype)

## 7. Change storage, not model code

The in-memory dataset can be copied to persistent Zarr storage and reopened
through the same API. Zarr is the default whenever a path is supplied.

In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory

with TemporaryDirectory() as directory:
    path = Path(directory) / "rollouts"
    persisted = dataset.copy_to(path)
    persisted.close()
    del persisted

    reopened = Dataset.open(path)
    summary = {
        "backend": reopened.backend.name,
        "episodes": reopened.num_episodes,
        "first_episode_length": len(reopened.episode(0)),
    }
    reopened.close()
    del reopened

summary

## Next steps

- [Getting started](../docs/getting_started.md) covers collection adapters,
  action-out sources, padding, and TensorDict conversion.
- [API reference](../docs/api.md) lists public signatures and contracts.
- The [README](../README.md) explains the repository's scope and philosophy.